# Skin Cancer Classification

Bu projece deri kanseri algilayan bir model gelistiriyoruz


In [3]:
import cv2
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split

from keras.models import Sequential
from keras.layers import Conv2D, Dense, Flatten, Input, MaxPooling2D, Dropout, BatchNormalization, Reshape

In [4]:
labels=['Cancer','Non_Cancer']
img_path='Skin_Data/'

In [5]:
img_list=[]
label_list=[]
for label in labels:
    for img_file in os.listdir(img_path+label):
        img_list.append(img_path+label+"/"+img_file)
        label_list.append(label)

In [6]:
df=pd.DataFrame({'img':img_list,'label':label_list})

In [7]:
df.head()

,img,label
0,Skin_Data/Cancer/1007-1.jpg,Cancer
1,Skin_Data/Cancer/1010-01.JPG,Cancer
2,Skin_Data/Cancer/1012-2.JPG,Cancer
3,Skin_Data/Cancer/1031-1.jpg,Cancer
4,Skin_Data/Cancer/1051-3(94).jpg,Cancer


In [8]:
df.tail()

,img,label
283,Skin_Data/Non_Cancer/953-1.JPG,Non_Cancer
284,Skin_Data/Non_Cancer/954-3.JPG,Non_Cancer
285,Skin_Data/Non_Cancer/955.JPG,Non_Cancer
286,Skin_Data/Non_Cancer/984.JPG,Non_Cancer
287,Skin_Data/Non_Cancer/986-1.JPG,Non_Cancer


In [9]:
d={'Cancer':1,'Non_Cancer':0}
df['encode_label']=df['label'].map(d)

In [10]:
df.head(1)

,img,label,encode_label
0,Skin_Data/Cancer/1007-1.jpg,Cancer,1


In [11]:
df.tail(2)

,img,label,encode_label
286,Skin_Data/Non_Cancer/984.JPG,Non_Cancer,0
287,Skin_Data/Non_Cancer/986-1.JPG,Non_Cancer,0


In [12]:
x=[] # bos liste
for img in df['img']:                 #her bir resm oku
    img=cv2.imread(str(img))
    img=cv2.resize(img,(170,170))
    img=img/255.0  # normalize et
    x.append(img) # listeye ekle

In [13]:
x=np.array(x) # rakam ve diziye cevir

In [14]:
y=df['encode_label']

In [15]:
x_train,x_test,y_train,y_test=train_test_split(x,y, random_state=42, test_size=0.20)

In [16]:
y_train = np.array(y_train, dtype=np.int32)
y_test = np.array(y_test, dtype=np.int32)

In [17]:
model = Sequential()
model.add(Input(shape=(170, 170, 3)))
model.add(Conv2D(32, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Conv2D(64, kernel_size=(3, 3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2, 2)))
model.add(Flatten())
model.add(Dense(128, activation='relu'))  # Added activation function
model.add(Dense(1, activation='sigmoid'))  # Ensure 2 classes for binary classification

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


In [ ]:
history=model.fit(x_train,y_train,validation_data=(x_test,y_test), epochs=20,verbose=1)

In [ ]:
model.save('skin_cancer_model.h5')

# Transfer Learning

In [ ]:
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
data_dir='Skin_Data'
img_width,img_heigth=224,224

train_datagen=ImageDataGenerator(rescale=1/255, validation_split=0.20)

train_datagenerator=train_datagen.flow_from_directory(directory=data_dir,target_size=(img_width,img_heigth),
                                class_mode='binary', subset='training')

test_datagen=ImageDataGenerator(rescale=1/255)
test_datagenerator=train_datagen.flow_from_directory(directory=data_dir,target_size=(img_width,img_heigth),
                                class_mode='binary', subset='validation')

base_model=VGG16(weights='imagenet', input_shape=(img_width,img_heigth,3),include_top=False)

model=Sequential()
model.add(base_model)
for layer in base_model.layers:
    layer.trainable=False
    
model.add(Flatten())
model.add(Dense(1024,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

model.fit(train_datagenerator,epochs=10,validation_data=test_datagenerator)


In [ ]:
model.summary()

In [ ]:
model.save('skin_cancer_TLv2.h5')

In [ ]:
#Transfer Learning

In [ ]:
from tensorflow.keras.applications.resnet50 import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input, decode_predictions

In [ ]:
from IPython.display import Image

In [ ]:
Image('savasucagi.jpg')

In [ ]:
img=image.load_img('savasucagi.jpg',target_size=(224,224))
img=image.img_to_array(img)
img=np.expand_dims(img,axis=0)
img=preprocess_input(img)

In [ ]:
model=ResNet50(weights='imagenet')

In [ ]:
pred=model.predict(img)

In [ ]:
decode_predictions(pred, top=1)